In [11]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from PIL import Image
from torchvision import models, transforms
from torchvision.transforms.functional import InterpolationMode

import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [12]:
DATA_ROOT = Path(r"/home/sergiocloudwork/dataset/sampled_1000/val")
DATA_ROOT = Path(r"D:/dataset/sampled_1000/val")
# number of image
z=10

In [13]:
loc = Path.cwd()
imgNetF = loc / "resNet_1000_data"

pth_file=[]
# Find and read all JSON files recursively
for file_path in imgNetF.rglob("*.pth"):
    print(f"Found: {file_path}")
    pth_file.append(file_path)


model_name={"inat_resnet18_1000_ptclasses_imagenet.pth":"ResNet18 pretrained model",
            "inat_resnet50_1000_ptclasses_imagenet.pth":"ResNet50 pretrained model",
            "inat_resnet18_1000_npclasses_imagenet.pth":"ResNet18 random start model",
            "inat_resnet50_1000_npclasses_imagenet.pth":"ResNet50 random start model"
            }

assert len(model_name.keys())==len(pth_file)

Found: c:\Users\mark\OneDrive\文档\GitHub\COMP9517GROUP_SUBJECT\src\performanceCompare\resNet_1000_data\inat_resnet18_1000_npclasses_imagenet.pth
Found: c:\Users\mark\OneDrive\文档\GitHub\COMP9517GROUP_SUBJECT\src\performanceCompare\resNet_1000_data\inat_resnet18_1000_ptclasses_imagenet.pth
Found: c:\Users\mark\OneDrive\文档\GitHub\COMP9517GROUP_SUBJECT\src\performanceCompare\resNet_1000_data\inat_resnet50_1000_npclasses_imagenet.pth
Found: c:\Users\mark\OneDrive\文档\GitHub\COMP9517GROUP_SUBJECT\src\performanceCompare\resNet_1000_data\inat_resnet50_1000_ptclasses_imagenet.pth


In [14]:
def create_model(model_name, num_classes):
    name = model_name.lower()

    if "resnet18" in name:
        model = models.resnet18(weights=None)
        model.fc = nn.Linear(
        model.fc.in_features,
        num_classes
    )
    elif "resnet50" in name:
        model = models.resnet50(weights=None)
        model.fc = nn.Linear(
        model.fc.in_features,
        num_classes
    )
    elif "alexnet" in name:
        model = models.alexnet(weights=None)
    else:
        raise ValueError(
            f"Cannot determine architecture from: {model_name}"
        )

    return model

In [15]:
def load_checkpoint_model(checkpoint_path, model_name):
    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=False
    )

    classes = checkpoint["classes"]
    num_classes = checkpoint["num_classes"]

    model = create_model(
        model_name=model_name,
        num_classes=num_classes
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    model = model.to(device)
    model.eval()

    return model, classes

In [16]:
gradcam_transform = transforms.Compose([
    transforms.Resize(
        (320, 320),
        interpolation=InterpolationMode.BILINEAR,
        antialias=True
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [17]:
def generate_gradcam(model,image_tensor,target_layer,target_class=None):
    """Generate a Grad-CAM heatmap for one image."""
    if image_tensor.ndim != 4 or image_tensor.size(0) != 1:
        raise ValueError(
            "image_tensor must have shape [1, C, H, W]"
        )

    activations = {}
    gradients = {}

    def forward_hook(module, inputs, output):
        activations["value"] = output

        def save_gradient(gradient):
            gradients["value"] = gradient

        output.register_hook(save_gradient)

    hook_handle = target_layer.register_forward_hook(forward_hook)

    try:
        model.zero_grad(set_to_none=True)

        # Grad-CAM needs gradients, so do not use no_grad/inference_mode here.
        with torch.enable_grad():
            logits = model(image_tensor)
            predicted_class = int(logits.argmax(dim=1).item())

            if target_class is None:
                target_class = predicted_class
            else:
                target_class = int(target_class)

            if not 0 <= target_class < logits.size(1):
                raise ValueError(
                    f"target_class must be between 0 and {logits.size(1) - 1}"
                )

            logits[0, target_class].backward()

        feature_maps = activations["value"][0]
        feature_gradients = gradients["value"][0]

        # One importance weight for each feature-map channel.
        weights = feature_gradients.mean(
            dim=(1, 2),
            keepdim=True
        )

        heatmap = (weights * feature_maps).sum(dim=0)
        heatmap = torch.relu(heatmap)

        heatmap = heatmap - heatmap.min()
        heatmap_max = heatmap.max()

        if heatmap_max.item() > 0:
            heatmap = heatmap / heatmap_max

        probabilities = torch.softmax(logits.detach(), dim=1)

        return {
            "heatmap": heatmap.detach().cpu().numpy(),
            "predicted_class": predicted_class,
            "predicted_probability": probabilities[
                0, predicted_class
            ].item(),
            "target_class": target_class,
            "target_probability": probabilities[
                0, target_class
            ].item()
        }

    finally:
        hook_handle.remove()

In [18]:
def process_one_model(checkpoint_path,model_name,image_tensor):
    model, classes = load_checkpoint_model(
        checkpoint_path,
        model_name
    )

    result = generate_gradcam(
        model=model,
        image_tensor=image_tensor,
        target_layer=model.layer4[-1]
    )

    
    result["model_name"] = model_name
    result["checkpoint_path"] = str(checkpoint_path)
    result["predicted_name"] = classes[
        result["predicted_class"]
    ]

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result

In [19]:
def one_img_prediction(img_path, output_dir):
    model_jobs = []

    
    image_path=Path(img_path)

    original_image = Image.open(image_path).convert("RGB")

    # previously Grad-CAM transform
    image_tensor = gradcam_transform(original_image)
    image_tensor = image_tensor.unsqueeze(0).to(device)



    for checkpoint_path in pth_file:
        filename = Path(checkpoint_path).name

        model_jobs.append({
            "path": checkpoint_path,
            "name": model_name[filename]
        })


    all_results = []

    for job in model_jobs:
        # print(f"\nProcessing: {job['name']}")

        result = process_one_model(
            checkpoint_path=job["path"],
            model_name=job["name"],
            image_tensor=image_tensor
        )

        all_results.append(result)

        print("Prediction:", result["predicted_name"])
        print(
            "Confidence:",
            f"{result['predicted_probability']:.2%}"
        )

    display_image = original_image.resize((320, 320))
    display_array = (
        np.asarray(display_image).astype(np.float32) / 255.0
    )

    fig, axes = plt.subplots(
        1,
        len(all_results),
        figsize=(6 * len(all_results), 6)
    )

    if len(all_results) == 1:
        axes = [axes]

    for ax, result in zip(axes, all_results):
        ax.imshow(display_array)
        ax.imshow(
            result["heatmap"],
            cmap="jet",
            alpha=0.4,
            extent=(0, 320, 320, 0)
        )

        ax.set_title(
            f"{result['model_name']}\n"
            f"{result['predicted_name'][:30]} "
            f"({result['predicted_probability']:.2%})"
        )
        ax.axis("off")
    
    fig.savefig(
        output_dir / image_path.name,
        dpi=300,
        bbox_inches="tight"
    )
    plt.tight_layout()
    plt.close(fig)
    plt.show()

In [20]:
Path.cwd()
output_dir = Path.cwd()/ "resnet_1000_output"
output_dir.mkdir(parents=True, exist_ok=True)

rng = random.Random()

#grab folder name
class_folders = [
    folder for folder in DATA_ROOT.iterdir()
    if folder.is_dir()
]

if z > len(class_folders):
    raise ValueError("too much")

selected_classes = rng.sample(class_folders, z)

selected_images = []

i=0
for class_folder in selected_classes:
    image_files = [
        file for file in class_folder.iterdir()
        if file.suffix.lower() in {".jpg"}
    ]
    i+=1
    print(i,'/',z)
    print(class_folder)

    image_path = rng.choice(image_files)
    print(image_path)

    one_img_prediction(image_path,output_dir)
    


1 / 10
D:\dataset\sampled_1000\val\02914_Animalia_Chordata_Actinopterygii_Siluriformes_Plotosidae_Plotosus_lineatus
D:\dataset\sampled_1000\val\02914_Animalia_Chordata_Actinopterygii_Siluriformes_Plotosidae_Plotosus_lineatus\69ea2b7d-4fd7-4d2f-a0fb-37fcef3e879c.jpg
Prediction: 04601_Animalia_Chordata_Elasmobranchii_Myliobatiformes_Aetobatidae_Aetobatus_narinari
Confidence: 53.97%
Prediction: 02864_Animalia_Chordata_Actinopterygii_Perciformes_Pomacentridae_Abudefduf_sexfasciatus
Confidence: 32.53%
Prediction: 04601_Animalia_Chordata_Elasmobranchii_Myliobatiformes_Aetobatidae_Aetobatus_narinari
Confidence: 30.95%
Prediction: 02921_Animalia_Chordata_Actinopterygii_Syngnathiformes_Syngnathidae_Phycodurus_eques
Confidence: 32.02%
2 / 10
D:\dataset\sampled_1000\val\08541_Plantae_Tracheophyta_Magnoliopsida_Lamiales_Martyniaceae_Proboscidea_louisianica
D:\dataset\sampled_1000\val\08541_Plantae_Tracheophyta_Magnoliopsida_Lamiales_Martyniaceae_Proboscidea_louisianica\cf5167b5-0d00-4c14-b79b-8ad8